# Комбинация: Debate + Voting — оценка стартапа

Три независимые команды дебатирующих агентов оценивают стартап. Каждая команда состоит из трёх ролей: оптимист, пессимист и медиатор. Оптимист ищет сильные стороны, пессимист — риски, медиатор выносит вердикт. Три вердикта агрегируются голосованием по правилу большинства (2 из 3).

Это максимальная защита от двух типов ошибок: дебаты внутри каждой команды снижают предвзятость, а голосование между командами снижает случайность. Стоимость — 9 вызовов LLM (3 команды × 3 агента) плюс голосование. Это оправдано для инвестиционных решений, где ошибка стоит миллионы.

```mermaid
graph LR
    START((START)) -->|Send x3| team1(["🔹 Команда 1<br/><i>обсуждает тему</i>"])
    START -->|Send x3| team2(["🔹 Команда 2<br/><i>обсуждает тему</i>"])
    START -->|Send x3| team3(["🔹 Команда 3<br/><i>обсуждает тему</i>"])
    team1 -->|вердикт| vote(["📊 Голосование<br/><i>подсчёт голосов</i>"])
    team2 -->|вердикт| vote
    team3 -->|вердикт| vote
    vote --> END((END))

    classDef coordinator fill:#4A90D9,stroke:#2C5F8A,color:#fff,stroke-width:2px
    classDef worker fill:#2ECC71,stroke:#1A8B4C,color:#fff,stroke-width:2px
    classDef aggregator fill:#F59E0B,stroke:#D97706,color:#fff,stroke-width:2px
    classDef terminal fill:#95A5A6,stroke:#707B7C,color:#fff

    class START,END terminal
    class team1,team2,team3 worker
    class vote aggregator
```

In [1]:
import sys
sys.path.insert(0, "..")

import operator
from typing import TypedDict, Annotated
from collections import Counter

from langgraph.graph import StateGraph, START, END
from langgraph.types import Send
from langchain_core.messages import HumanMessage, SystemMessage
from llm_config import get_llm, check_api_key

In [2]:
if not check_api_key():
    raise ValueError("API key is not set")
else:
    print("API key is set")

API key is set


## Два класса состояний

Системе нужны два типа состояний. `StartupEvalState` — глобальное состояние всего графа: описание стартапа, список вердиктов от команд (с редьюсером `operator.add` для автоматического объединения) и итоговое решение. `DebateTeamInput` — локальное состояние для одной дебатной команды, которое передаётся через `Send`: описание стартапа и номер команды.

In [3]:
llm = get_llm()

class StartupEvalState(TypedDict):
    startup: str                                       # Описание стартапа
    team_verdicts: Annotated[list[str], operator.add]  # Вердикты команд
    final_decision: str                                # Итоговое решение

class DebateTeamInput(TypedDict):
    startup: str
    team_id: int

## Дебатная команда

Каждая команда — три последовательных вызова LLM с разными ролями. Оптимист находит сильные стороны стартапа и потенциал роста. Пессимист получает аргументы оптимиста и ищет слабости, риски, причины не инвестировать. Медиатор выслушивает обоих и выносит вердикт строго одним словом: INVEST, PASS или CONDITIONAL. Такая структура гарантирует, что каждый вердикт — результат рассмотрения обеих сторон.

In [4]:
def debate_team_node(state: DebateTeamInput) -> dict:
    """Одна команда из трёх дебатирующих агентов."""
    startup = state["startup"]
    team_id = state["team_id"]

    # Оптимист
    optimist_resp = llm.invoke([
        SystemMessage(content=f"Ты — оптимист в команде #{team_id}. Найди сильные стороны стартапа и потенциал роста."),
        HumanMessage(content=startup)
    ])
    print(f"  [Команда {team_id}, оптимист] {optimist_resp.content[:80]}...")

    # Пессимист — получает аргументы оптимиста
    pessimist_resp = llm.invoke([
        SystemMessage(content=f"Ты — пессимист в команде #{team_id}. Найди слабости, риски, причины не инвестировать."),
        HumanMessage(content=f"{startup}\n\nАргументы оптимиста: {optimist_resp.content}")
    ])
    print(f"  [Команда {team_id}, пессимист] {pessimist_resp.content[:80]}...")

    # Медиатор — выносит вердикт
    mediator_resp = llm.invoke([
        SystemMessage(content=f"""Ты — медиатор в команде #{team_id}. Выслушал оптимиста и пессимиста.
Вынеси вердикт строго одним словом из трёх: INVEST, PASS или CONDITIONAL.
Затем кратко обоснуй."""),
        HumanMessage(content=f"Оптимист: {optimist_resp.content}\nПессимист: {pessimist_resp.content}")
    ])
    print(f"  [Команда {team_id}, медиатор] {mediator_resp.content[:80]}...")

    return {"team_verdicts": [f"[Команда {team_id}]: {mediator_resp.content}"]}

## Fan-out через Send и голосование

Функция `fan_out_debate_teams` возвращает список из трёх объектов `Send`, каждый из которых направляет выполнение к одному и тому же узлу `debate_team`, но с разным `team_id`. LangGraph запускает все три команды параллельно и собирает их результаты через редьюсер `operator.add` в поле `team_verdicts`.

После того как все три команды завершили дебаты, узел голосования получает три вердикта и применяет правило большинства. LLM подсчитывает голоса (INVEST / PASS / CONDITIONAL) и формулирует итоговое решение с обоснованием.

In [5]:
def fan_out_debate_teams(state: StartupEvalState) -> list[Send]:
    """Запуск трёх независимых дебатных команд."""
    print("  Запускаю 3 независимые команды дебатов")
    return [
        Send("debate_team", {"startup": state["startup"], "team_id": i})
        for i in range(1, 4)
    ]

def vote_on_verdicts(state: StartupEvalState) -> dict:
    """Голосование по вердиктам трёх команд."""
    verdicts_text = "\n".join(state["team_verdicts"])

    response = llm.invoke([
        SystemMessage(content="""Перед тобой вердикты трёх независимых экспертных команд.
Подсчитай голоса: INVEST, PASS, CONDITIONAL.
Примени правило большинства (2 из 3).
Объясни итоговое решение."""),
        HumanMessage(content=verdicts_text)
    ])
    print(f"  [Голосование] {response.content[:100]}...")
    return {"final_decision": response.content}

## Сборка графа

Граф содержит всего два узла: `debate_team` и `vote`. Условное ребро из `START` через `fan_out_debate_teams` создаёт три параллельных экземпляра `debate_team` с помощью `Send`. После завершения всех трёх команд LangGraph автоматически объединяет их результаты в `team_verdicts` и передаёт управление узлу `vote`.

In [6]:
graph = StateGraph(StartupEvalState)

graph.add_node("debate_team", debate_team_node)
graph.add_node("vote", vote_on_verdicts)

graph.add_conditional_edges(START, fan_out_debate_teams)
graph.add_edge("debate_team", "vote")
graph.add_edge("vote", END)

app = graph.compile()

## Запуск

Передаём описание стартапа для оценки. Три команды параллельно проведут внутренние дебаты, каждая вынесет вердикт, затем голосование определит итоговое решение.

In [7]:
result = app.invoke(
    {
        "startup": "AI-стартап: мультиагентная платформа для автоматизации юридических процессов. "
                   "Seed-раунд, запрашивают $2M, команда из 3 человек (CEO с опытом в legal-tech, "
                   "CTO из Google DeepMind, lead engineer из LangChain).",
        "team_verdicts": [],
        "final_decision": "",
    },
    config={"recursion_limit": 30},
)

print("\n" + "=" * 60)
print("ИТОГОВОЕ РЕШЕНИЕ:")
print("=" * 60)
print(result["final_decision"])

  Запускаю 3 независимые команды дебатов


  [Команда 3, оптимист] Сильный кейс для #3 — здесь есть, за что зацепиться.

### Что выглядит особенно ...


  [Команда 1, оптимист] У стартапа есть **очень сильный инвестиционный профиль**, особенно для seed-стад...


  [Команда 2, оптимист] У стартапа очень сильный профиль для seed-раунда. Если смотреть как оптимист ком...


  [Команда 3, пессимист] Позиция пессимиста: я бы **не спешил инвестировать**. На бумаге это красиво, но ...
  [Команда 2, пессимист] С позиции пессимиста команды #2 я бы **не спешил инвестировать**. На бумаге всё ...


  [Команда 2, медиатор] CONDITIONAL

Команда и рынок выглядят сильными, но для уверенного INVEST не хват...
  [Команда 3, медиатор] CONDITIONAL

Команда сильная и рынок дорогой, но пока это скорее ставка на потен...


  [Команда 1, пессимист] Ниже — взгляд **пессимиста**: почему в эту историю **можно не инвестировать сейч...


  [Команда 1, медиатор] CONDITIONAL

Команда и рынок выглядят сильными, но без подтвержденного traction ...


  [Голосование] Подсчёт голосов:

- **INVEST**: 0
- **PASS**: 0
- **CONDITIONAL**: 3

### Итог по правилу большинств...

ИТОГОВОЕ РЕШЕНИЕ:
Подсчёт голосов:

- **INVEST**: 0
- **PASS**: 0
- **CONDITIONAL**: 3

### Итог по правилу большинства (2 из 3)
**CONDITIONAL**

### Почему
Все три независимые команды пришли к одному и тому же выводу:  
команда и рынок выглядят сильными, но **не хватает подтверждённого traction** — нет достаточных доказательств:
- платящих клиентов,
- повторяемого use case,
- ясного wedge / GTM,
- понятного moat / защиты от копирования,
- подтверждённого ROI.

То есть проект выглядит **перспективным**, но пока это скорее **ставка на потенциал**, чем уже доказанный инвестиционный кейс. Поэтому общий вердикт — **инвестировать только при выполнении дополнительных условий**, то есть **CONDITIONAL**.


## Итог

Комбинация Debate + Voting объединяет два паттерна для максимальной надёжности решений:

- **Debate** внутри каждой команды (оптимист, пессимист, медиатор) устраняет предвзятость: решение не может быть принято без рассмотрения обеих сторон.
- **Voting** между тремя независимыми командами устраняет случайность: одна ошибочная команда не определяет итог.
- **Send** обеспечивает параллельный запуск команд — три дебата идут одновременно, а LangGraph автоматически собирает результаты через редьюсер.

Стоимость (9 вызовов LLM + голосование) оправдана для задач, где цена ошибки высока: инвестиционные решения, медицинская диагностика, юридическая экспертиза.